# Домашнее задание 3. Распознавание именованных сущностей
Блок 2 «Transformer и pretrained models»
Распознавание сущностей стоит за многими прикладными задачами: обезличивание персональных данных
в документах, поиск контрагентов в договорах, разметка новостей по компаниям и людям. А ещё это первая
домашка, где вы сами дообучаете трансформер через `Trainer`. Цикл «токенизация → обучение → оценка»
здесь тот же, что для любой задачи в Hugging Face.


При классификации текста метка одна на всё предложение. В распознавании сущностей метка нужна
**каждому слову**: персона (`PER`), организация (`ORG`), место (`LOC`) или прочее (`MISC`) в разметке BIO.
Главная сложность — токенизатор режет слова на части, а метки заданы для слов.

Данные — русская часть WikiNEuRal (Tedeschi et al., 2021): предложения из Википедии с автоматически
полученной и отфильтрованной разметкой.

| Задание | Баллы |
|---|---:|
| 1. Знакомство с данными | 1 |
| 2. Выравнивание меток с подтокенами | 2,5 |
| 3. Метрики на уровне сущностей | 1,5 |
| 4. Дообучение rubert-tiny2 | 2 |
| 5. Сравнение с готовой многоязычной моделью | 1,5 |
| 6. Анализ ошибок и выводы | 1,5 |
| **Итого** | **10** |

**Как сдавать.** Почти весь код уже написан. В каждом задании с кодом есть одна функция или ячейка
с 3–5 пропусками: на месте `...` нужно дописать код, что именно — сказано в комментарии `TODO`.
Выполните ноутбук целиком, сдайте `.ipynb`
с выводами. На вопросы хватит одного-двух предложений.
Обучение на CPU занимает около 10 минут, на GPU или Apple Silicon — 2–3 минуты.

In [ ]:
# requirements (Python 3.10–3.12); в скобках — версии, на которых ноутбук прогнан:
#   torch>=2.4           (проверено 2.15.0)
#   transformers>=4.46   (проверено 5.22.2)
#   datasets>=3.0        (проверено 5.1.0)
#   accelerate>=0.34     (проверено 1.15.0)
#   seqeval>=1.2         (проверено 1.2.2)
#   pandas>=2.2          (проверено 3.1.1)
#   numpy>=1.26          (проверено 2.5.4)
#   matplotlib>=3.8      (проверено 3.11.2)
# Установка локально: pip install -r requirements.txt
# Google Colab: раскомментируйте строку ниже и перезапустите среду после установки.
# %pip install -q "torch>=2.4" "transformers>=4.46" "datasets>=3.0" "accelerate>=0.34" "seqeval>=1.2" "pandas>=2.2" "numpy>=1.26" "matplotlib>=3.8"

import sys
print("Python", sys.version.split()[0])

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from collections import Counter
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer,
                          DataCollatorForTokenClassification, set_seed)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

SEED = 42
set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
WORK_DIR = Path("hw3_work"); WORK_DIR.mkdir(exist_ok=True)
LABELS = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = dict(enumerate(LABELS))
print("device:", DEVICE)

## Данные (готовый код)
Подвыборка нужна, чтобы обучение шло минуты даже на CPU.

Берём подвыборку: 8 000 предложений для обучения, по 1 000 для проверки и теста.

In [ ]:
files = {split: f"data/{split}_ru-00000-of-00001.parquet" for split in ("train", "val", "test")}
full = load_dataset("Babelscape/wikineural", data_files=files)
ds = {
    "train": full["train"].shuffle(seed=SEED).select(range(8000)),
    "val": full["val"].shuffle(seed=SEED).select(range(1000)),
    "test": full["test"].shuffle(seed=SEED).select(range(1000)),
}
example = ds["train"][3]
display(pd.DataFrame({"token": example["tokens"], "tag": [LABELS[t] for t in example["ner_tags"]]}).T)

## Задание 1. Знакомство с данными (1 балл)
Перед обучением всегда смотрим на данные. Если какой-то тип сущностей редкий, модель будет на нём
ошибаться, и это нужно знать заранее, чтобы правильно читать метрики.

Посчитайте на train число сущностей каждого типа (сущность начинается с `B-`) и долю предложений
без сущностей. Соберите и список длин предложений в словах — гистограмма по нему строится готовым кодом. Какой тип самый редкий?

In [ ]:
entity_counts = ...  # TODO: Counter типов LABELS[t][2:] по всем тегам t всех row["ner_tags"] в ds["train"],
                     #       где LABELS[t] начинается с "B-"
share_without_entities = ...  # TODO: np.mean от списка флагов «все теги предложения равны 0» по ds["train"]
display(pd.Series(entity_counts, name="сущностей в train").sort_values(ascending=False).to_frame())
print(f"Предложений без сущностей: {share_without_entities:.1%}")
lengths = ...  # TODO: список длин предложений train в словах: len(row["tokens"]) для каждой row
plt.figure(figsize=(7, 3)); plt.hist(lengths, bins=40, color="#bd9fe3")
plt.xlabel("слов в предложении"); plt.ylabel("предложений"); plt.title("Длина предложений, train"); plt.show()
print("Медиана:", int(np.median(lengths)), "| 99-й перцентиль:", int(np.percentile(lengths, 99)))

## Задание 2. Выравнивание меток с подтокенами (2,5 балла)
Это главная техническая ловушка NER на трансформерах. Токенизатор режет «Петербурге» на несколько
кусочков, метки заданы для слов, а модель предсказывает их для подтокенов. Ошибка в выравнивании
не вызывает исключений: модель тихо учится на сдвинутых метках и показывает плохое качество.

Токенизатор вызывается с `is_split_into_words=True`, а `encoding.word_ids(i)` говорит, какому слову
принадлежит каждый подтокен (`None` — для служебных `[CLS]`, `[SEP]`).

Допишите в `tokenize_and_align_labels(batch, label_all_tokens=False)` три метки:

* служебные токены получают метку `-100` (игнорируется функцией потерь);
* первый подтокен слова получает метку слова;
* остальные подтокены слова: при `label_all_tokens=False` — `-100`; при `True` — метку слова,
  но `B-X` заменяется на `I-X` (продолжение сущности);
* обрезка по `max_length=128`.

In [ ]:
MODEL_ID = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
B_TO_I = {label2id[l]: label2id["I-" + l[2:]] for l in LABELS if l.startswith("B-")}   # id B-X → id I-X

def tokenize_and_align_labels(batch, label_all_tokens=False):
    encoding = tokenizer(batch["tokens"], is_split_into_words=True, truncation=True, max_length=128)
    all_labels = []
    for i, word_tags in enumerate(batch["ner_tags"]):
        labels, previous = [], None
        for word_id in encoding.word_ids(batch_index=i):
            if word_id is None:
                labels.append(...)  # TODO: служебный токен — метка -100
            elif word_id != previous:
                labels.append(...)  # TODO: первый подтокен слова — метка слова word_tags[word_id]
            else:
                tag = word_tags[word_id]
                labels.append(...)  # TODO: B_TO_I.get(tag, tag), если label_all_tokens, иначе -100
            previous = word_id
        all_labels.append(labels)
    encoding["labels"] = all_labels
    return encoding

demo = {"tokens": [["Анна", "Ахматова", "жила", "в", "Петербурге"]], "ner_tags": [[1, 2, 0, 0, 5]]}
enc = tokenize_and_align_labels(demo)
print(list(zip(tokenizer.convert_ids_to_tokens(enc["input_ids"][0]), enc["labels"][0])))

Проверки ловят типичные ошибки выравнивания до того, как вы потратите 10 минут на обучение.

In [ ]:
# Проверки (не меняйте)
enc_first = tokenize_and_align_labels(demo)
enc_all = tokenize_and_align_labels(demo, label_all_tokens=True)
word_ids = enc_first.word_ids(0)
assert enc_first["labels"][0][0] == -100 and enc_first["labels"][0][-1] == -100
first_positions = [j for j, w in enumerate(word_ids) if w is not None and (j == 0 or word_ids[j - 1] != w)]
assert [enc_first["labels"][0][j] for j in first_positions] == demo["ner_tags"][0]
assert all(l == -100 for j, l in enumerate(enc_first["labels"][0]) if j not in first_positions)
assert all(l != 1 for j, l in enumerate(enc_all["labels"][0]) if j not in first_positions and word_ids[j] == 0)
print("Проверки пройдены")

tokenized = {split: d.map(tokenize_and_align_labels, batched=True, remove_columns=d.column_names)
             for split, d in ds.items()}
print(tokenized["train"])

## Задание 3. Метрики на уровне сущностей (1,5 балла)
`Trainer` будет вызывать эту функцию после каждой эпохи — так мы увидим, учится ли модель.

Точность по токенам здесь обманчива: большинство меток — `O`. Используем `seqeval`: сущность считается
найденной, только если совпали и границы, и тип.

Допишите `compute_metrics(eval_pred)` для `Trainer`: из логитов взять `argmax`, выбросить позиции
с меткой `-100`, перевести номера в строки меток и вернуть `precision`, `recall`, `f1`
(микро-усреднение по сущностям).

In [ ]:
def to_tag_sequences(predictions, labels):
    """Убрать позиции -100 и перевести id в строки. Возвращает (true_seqs, pred_seqs)."""
    true_seqs, pred_seqs = [], []
    for pred_row, label_row in zip(predictions, labels):
        keep = ...  # TODO: список пар (p, l) из zip(pred_row, label_row), где l != -100
        true_seqs.append([LABELS[l] for _, l in keep])
        pred_seqs.append([LABELS[p] for p, _ in keep])
    return true_seqs, pred_seqs

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    true_seqs, pred_seqs = ...  # TODO: to_tag_sequences от logits.argmax(-1) и labels
    return {"precision": precision_score(true_seqs, pred_seqs),
            "recall": recall_score(true_seqs, pred_seqs),
            "f1": ...}  # TODO: f1_score от тех же последовательностей

toy_pred = np.array([[0, 1, 2, 0, 5, 0]]); toy_lab = np.array([[-100, 1, 2, 0, 6, -100]])
t, p = to_tag_sequences(toy_pred, toy_lab)
assert t == [["B-PER", "I-PER", "O", "I-LOC"]] and p == [["B-PER", "I-PER", "O", "B-LOC"]]

## Задание 4. Дообучение (2 балла)
`rubert-tiny2` — маленькая русская модель: на CPU она учится за минуты. `Trainer` берёт на себя
цикл обучения, батчи и оценку, а нам остаётся задать гиперпараметры.

Допишите в `TrainingArguments` четыре параметра: 3 эпохи, `learning_rate=1e-4`, оценка на `val`
после каждой эпохи, без сохранения контрольных точек. Остальное (пакет 32, `weight_decay=0.01`,
разогрев 10% шагов) уже задано.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(MODEL_ID, num_labels=len(LABELS),
                                                        id2label=id2label, label2id=label2id)
collator = DataCollatorForTokenClassification(tokenizer)

args = TrainingArguments(output_dir=str(WORK_DIR / "rubert-ner"),
                         num_train_epochs=...,   # TODO: 3 эпохи
                         learning_rate=...,      # TODO: 1e-4
                         per_device_train_batch_size=32, per_device_eval_batch_size=64, weight_decay=0.01,
                         warmup_steps=0.1,
                         eval_strategy=...,      # TODO: "epoch" — оценка после каждой эпохи
                         save_strategy=...,      # TODO: "no" — не сохранять контрольные точки
                         logging_steps=50, report_to="none", seed=SEED)

trainer = Trainer(model=model, args=args, train_dataset=tokenized["train"], eval_dataset=tokenized["val"],
                  data_collator=collator, processing_class=tokenizer, compute_metrics=compute_metrics)
t0 = perf_counter(); trainer.train(); print(f"Обучение: {perf_counter() - t0:.0f} с")

Смотрим метрики по эпохам: растёт ли F1 на val и не начинается ли переобучение (loss на val растёт,
хотя на train падает).

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
display(history.loc[history["eval_f1"].notna(), ["epoch", "eval_loss", "eval_precision", "eval_recall", "eval_f1"]].round(4))

## Задание 5. Сравнение с готовой моделью (1,5 балла)
В реальном проекте сначала проверяют, нет ли готовой модели. Здесь она есть, и она намного больше.
Вопрос не в том, кто точнее, а в том, стоит ли прирост качества замедления и затрат на железо.

`Babelscape/wikineural-multilingual-ner` — многоязычный BERT (около 177 млн параметров), дообученный
авторами датасета на всех языках WikiNEuRal. Сравним его с вашей моделью на одних и тех же 1 000
тестовых предложениях **на уровне слов**.

Допишите в `predict_words(model, tok, sentences, batch_size=32)` три строки: номера слов для подтокенов
(`word_ids`), `argmax` логитов и метку слова по его **первому** подтокену. Метки переводятся через `model.config.id2label` — у чужой
модели порядок меток может отличаться! F1 по типам, число параметров и скорость (предложений в секунду)
считаются готовым кодом.

In [ ]:
@torch.no_grad()
def predict_words(model, tok, sentences, batch_size=32):
    model.eval().to(DEVICE)
    id2l = model.config.id2label
    result = []
    for start in range(0, len(sentences), batch_size):
        chunk = sentences[start:start + batch_size]
        enc = tok(chunk, is_split_into_words=True, truncation=True, max_length=128,
                  padding=True, return_tensors="pt")
        word_ids = ...  # TODO: список enc.word_ids(i) для каждого предложения i батча
        preds = ...  # TODO: model(**{k: v.to(DEVICE) for k, v in enc.items()}).logits, затем .argmax(-1).cpu().numpy()
        for i, words in enumerate(chunk):
            tags = ["O"] * len(words)   # обрезанные по длине слова останутся "O"
            previous = None
            for j, w in enumerate(word_ids[i]):
                if w is not None and w != previous:
                    tags[w] = ...  # TODO: строковая метка id2l[int(preds[i, j])]
                previous = w
            result.append(tags)
    return result

test_sentences = [row["tokens"] for row in ds["test"]]
test_gold = [[LABELS[t] for t in row["ner_tags"]] for row in ds["test"]]

ref_id = "Babelscape/wikineural-multilingual-ner"
ref_tok = AutoTokenizer.from_pretrained(ref_id)
ref_model = AutoModelForTokenClassification.from_pretrained(ref_id)

systems, rows = {}, []
for name, (m, tk) in {"rubert-tiny2 (наше)": (trainer.model, tokenizer), "mBERT WikiNEuRal": (ref_model, ref_tok)}.items():
    predict_words(m, tk, test_sentences[:32])               # прогрев
    t0 = perf_counter()
    systems[name] = predict_words(m, tk, test_sentences)
    seconds = perf_counter() - t0
    rows.append({"система": name, "параметров, млн": round(sum(p.numel() for p in m.parameters()) / 1e6, 1),
                 "предложений/с": round(len(test_sentences) / seconds, 1),
                 "F1": round(f1_score(test_gold, systems[name]), 4)})
display(pd.DataFrame(rows))
per_type = {name: pd.DataFrame(classification_report(test_gold, pred, output_dict=True)).T["f1-score"]
            for name, pred in systems.items()}
display(pd.DataFrame(per_type).loc[["PER", "ORG", "LOC", "MISC", "micro avg"]].round(4))

**Вопрос 1.** Во сколько раз эталонная модель больше и медленнее вашей и на сколько выше её F1?

**Ответ:** …

## Задание 6. Анализ ошибок (1,5 балла)
Разные ошибки лечатся по-разному. Ошибки границ часто упираются в разметку, ошибки типа — в нехватку
контекста или данных, а лишние сущности — признак того, что модель цепляется за заглавные буквы.

1. Допишите `error_type(gold_entities, pred_entity)`, классифицирующую каждую **предсказанную**
   сущность вашей модели как `верно`, `тип` (границы совпали, тип нет), `границы` (пересекается
   с эталонной сущностью того же типа, но границы другие) или `лишняя` (не пересекается ни с одной).
   Сущность — кортеж `(тип, начало, конец)` в словах, конец не включается; используйте `get_entities`.
2. Распределение типов ошибок и 8 предложений, где модели расходятся, выводит готовый код.

In [ ]:
from seqeval.metrics.sequence_labeling import get_entities

def entities(tags):
    """Список (тип, начало, конец_исключительно)."""
    return [(t, s, e + 1) for t, s, e in get_entities(tags)]

def error_type(gold_entities, pred_entity):
    p_type, p_start, p_end = pred_entity
    if ...:  # TODO: предсказанная сущность целиком есть в gold_entities
        return "верно"
    for g_type, g_start, g_end in gold_entities:
        if ...:  # TODO: границы совпали: (g_start, g_end) == (p_start, p_end)
            return "тип"
    for g_type, g_start, g_end in gold_entities:
        if ...:  # TODO: тот же тип и отрезки пересекаются: p_start < g_end и g_start < p_end
            return "границы"
    return "лишняя"

assert error_type([("PER", 0, 2)], ("PER", 0, 2)) == "верно"
assert error_type([("PER", 0, 2)], ("ORG", 0, 2)) == "тип"
assert error_type([("PER", 0, 2)], ("PER", 1, 2)) == "границы"
assert error_type([("PER", 0, 2)], ("LOC", 5, 6)) == "лишняя"

Сравниваем с эталоном и с большой моделью. Где обе модели ошибаются одинаково, нередко виновата разметка.

In [ ]:
ours = systems["rubert-tiny2 (наше)"]
ref = systems["mBERT WikiNEuRal"]
kinds = Counter()
missed = 0
for gold, pred in zip(test_gold, ours):
    g = entities(gold)
    p = entities(pred)
    kinds.update(error_type(g, e) for e in p)
    missed += sum(1 for e in g if not any(e[1] < q[2] and q[1] < e[2] for q in p))
display(pd.Series(kinds, name="предсказанных сущностей").to_frame().assign(доля=lambda d: (d.iloc[:, 0] / d.iloc[:, 0].sum()).round(3)))
print("Эталонных сущностей, не задетых ни одним предсказанием:", missed)

def render(tokens, tags):
    return " ".join(f"[{tok}|{tag[2:]}]" if tag != "O" else tok for tok, tag in zip(tokens, tags))

disagree = [i for i in range(len(test_gold)) if ours[i] != ref[i]][:8]
for i in disagree:
    print("ЭТАЛОН:", render(test_sentences[i], test_gold[i]))
    print("НАША:  ", render(test_sentences[i], ours[i]))
    print("mBERT: ", render(test_sentences[i], ref[i]), "\n")

**Вопрос 2.** Какой тип ошибок у вашей модели преобладает и какая сущность (`PER`, `ORG`, `LOC`, `MISC`)
даётся хуже всего?

**Ответ:** …

**Дополнительно (без баллов).** Обучите модель с `label_all_tokens=True` и сравните F1. Объясните разницу.